In [9]:
import pandas as pd
import numpy as np
from pathlib import Path

# مسیر اصلی دیتاست (در روت پروژه)
dataset_dir = Path("../Dataset")

target_folders = [
    "ABSA_KPI",
    "Feature_KPI_output",
    "Cleaned_output",
]

print("🚀 شروع اسکن جامع کیفیت داده‌ها (Data Quality Auto-Scanner)...\n")

for folder_name in target_folders:
    folder_path = dataset_dir / folder_name

    if not folder_path.exists():
        continue

    print("\n" + "=" * 75)
    print(f"📂 در حال بازرسی پوشه: {folder_name}")
    print("=" * 75)

    # پیدا کردن تمام فایل‌های Parquet
    parquet_files = list(folder_path.rglob("*.parquet"))

    # برای جلوگیری از طولانی شدن خروجی در پوشه‌های پارت‌بندی‌شده،
    # فقط part_0000.parquet بررسی می‌شود.
    for file_path in parquet_files:

        if (
            "part_" in file_path.name
            and file_path.name != "part_0000.parquet"
        ):
            continue

        try:
            df = pd.read_parquet(file_path)

            if df.empty:
                print(
                    f"⚠️ فایل {file_path.parent.name}/{file_path.name} کاملاً خالی است!"
                )
                continue

            print(f"\n📄 گزارش فایل: {file_path.parent.name} / {file_path.name}")
            print(f"📏 ابعاد: {df.shape[0]:,} سطر و {df.shape[1]} ستون")

            # =====================================================
            # 1. بررسی مقادیر گمشده (Null / NaN)
            # =====================================================
            missing_stats = df.isna().sum()
            missing_cols = missing_stats[missing_stats > 0]

            if not missing_cols.empty:
                print("🚨 مقادیر گمشده (Null):")

                for col, count in missing_cols.items():
                    percent = (count / len(df)) * 100
                    print(
                        f"   - ستون '{col}': {count:,} رکورد ({percent:.1f}%)"
                    )
            else:
                print("✅ هیچ مقدار گمشده‌ای وجود ندارد.")

            # =====================================================
            # 2. بررسی ردیف‌های کاملاً تکراری
            # =====================================================
            duplicate_count = df.duplicated().sum()

            if duplicate_count > 0:
                print(
                    f"🔁 تعداد {duplicate_count:,} ردیف کاملاً تکراری یافت شد!"
                )

            # =====================================================
            # 3. بررسی مقادیر بی‌نهایت (Infinity)
            # =====================================================
            numeric_cols = df.select_dtypes(include=[np.number]).columns

            for col in numeric_cols:
                inf_count = np.isinf(df[col]).sum()

                if inf_count > 0:
                    print(
                        f"♾️ هشدار: ستون '{col}' دارای {inf_count:,} مقدار بی‌نهایت است!"
                    )

            # =====================================================
            # 4. بررسی قوانین منطقی (Logical Checks)
            # =====================================================

            # خرید نباید بیشتر از بازدید باشد
            if (
                "is_purchase" in df.columns
                and "is_view" in df.columns
            ):
                invalid_funnel = df[
                    df["is_purchase"] > df["is_view"]
                ]

                if not invalid_funnel.empty:
                    print(
                        f"❌ خطای منطقی: {len(invalid_funnel):,} رکورد، خرید بیشتر از بازدید دارند!"
                    )

            # امتیاز محصولات باید بین 0 تا 100 باشد
            if "Rate" in df.columns:
                invalid_product_rate = df[
                    (df["Rate"] < 0)
                    | (df["Rate"] > 100)
                ]

                if not invalid_product_rate.empty:
                    print(
                        f"❌ خطای منطقی: {len(invalid_product_rate):,} محصول با امتیاز نامعتبر (خارج از بازه 0 تا 100)!"
                    )

            # امتیاز کامنت‌ها باید بین 0 تا 5 باشد
            if "rate" in df.columns:
                invalid_comment_rate = df[
                    (df["rate"] < 0)
                    | (df["rate"] > 5)
                ]

                if not invalid_comment_rate.empty:
                    print(
                        f"❌ خطای منطقی: {len(invalid_comment_rate):,} کامنت با امتیاز نامعتبر (خارج از بازه 0 تا 5)!"
                    )

            # قیمت نباید منفی باشد
            if "Price" in df.columns:
                invalid_price = df[df["Price"] < 0]

                if not invalid_price.empty:
                    print(
                        f"❌ خطای منطقی: {len(invalid_price):,} محصول با قیمت منفی یافت شد!"
                    )

            # بررسی event_type
            if "event_type" in df.columns:
                valid_events = [
                    "view",
                    "add_to_cart",
                    "purchase",
                ]

                invalid_events = df[
                    ~df["event_type"].isin(valid_events)
                ]

                if not invalid_events.empty:
                    print(
                        f"❌ خطای منطقی: {len(invalid_events):,} رکورد با event_type ناشناخته!"
                    )

        except Exception as e:
            print(f"⚠️ خطا در پردازش {file_path.name}: {e}")

print("\n🎉 اسکنر کیفیت داده‌ها کار خود را با موفقیت به پایان رساند!")

🚀 شروع اسکن جامع کیفیت داده‌ها (Data Quality Auto-Scanner)...


📂 در حال بازرسی پوشه: ABSA_KPI

📄 گزارش فایل: ABSA_KPI / aspect_frequency.parquet
📏 ابعاد: 255,076 سطر و 2 ستون
✅ هیچ مقدار گمشده‌ای وجود ندارد.

📄 گزارش فایل: ABSA_KPI / aspect_kpi.parquet
📏 ابعاد: 255,076 سطر و 6 ستون
✅ هیچ مقدار گمشده‌ای وجود ندارد.

📄 گزارش فایل: ABSA_KPI / brand_kpi.parquet
📏 ابعاد: 6,641 سطر و 6 ستون
✅ هیچ مقدار گمشده‌ای وجود ندارد.

📄 گزارش فایل: ABSA_KPI / category_kpi.parquet
📏 ابعاد: 195 سطر و 6 ستون
✅ هیچ مقدار گمشده‌ای وجود ندارد.

📄 گزارش فایل: ABSA_KPI / product_aspect_kpi.parquet
📏 ابعاد: 2,129,962 سطر و 7 ستون
✅ هیچ مقدار گمشده‌ای وجود ندارد.

📄 گزارش فایل: ABSA_KPI / product_kpi.parquet
📏 ابعاد: 331,599 سطر و 8 ستون
🚨 مقادیر گمشده (Null):
   - ستون 'top_aspect': 60,005 رکورد (18.1%)
   - ستون 'worst_aspect': 169,050 رکورد (51.0%)

📄 گزارش فایل: ABSA_KPI / product_master_absa.parquet
📏 ابعاد: 948,352 سطر و 16 ستون
🚨 مقادیر گمشده (Null):
   - ستون 'Category2': 181,346 رکورد (19.1%)
   - ستون

In [ ]:
#برای اینکه ببینیم ایونت هامون چیان باتوجه به اینکه خطای بررسی کیفیت داده مون نشون داد یه ایونت جا مونده

import pandas as pd
from pathlib import Path

# خواندن فایل لاگ رفتار کاربران
# (از پوشه Feature_KPI_output)
df_logs = pd.read_parquet(
    "../Dataset/Feature_KPI_output/user_behavior_enriched.parquet"
)

# نمایش تعداد هر نوع رویداد موجود در ستون event_type
print(
    df_logs["event_type"].value_counts(dropna=False)
)

event_type
view                2706875
add_to_cart          596163
purchase             328291
remove_from_cart     119087
Name: count, dtype: int64


In [ ]:
#باتوجه به خروجی انالیز کیفیت داده میخوایم ببینیم واقعا محصولی هست که خریدش بیشتر از ویو هاش باشه یا نه


import pandas as pd
from pathlib import Path

# خواندن جدول ویژگی محصولات
# (یا جدول نهایی df_golden در صورت استفاده)
df_products = pd.read_parquet(
    "../Dataset/Feature_KPI_output/product_master.parquet"
)

# بررسی وجود ستون‌های مورد نیاز
if (
    "is_view" in df_products.columns
    and "is_purchase" in df_products.columns
):

    # محصولاتی که تعداد خریدشان از تعداد بازدید بیشتر است
    anomaly_products = df_products[
        df_products["is_purchase"] > df_products["is_view"]
    ]

    print(f"📊 تعداد کل محصولات بررسی شده: {len(df_products):,}")
    print(
        f"❌ تعداد محصولاتی که خرید بیشتر از بازدید داشتند: {len(anomaly_products):,}\n"
    )

    if not anomaly_products.empty:
        print("👀 نمونه‌ای از این محصولات:")

        # ستون‌هایی که می‌خواهیم نمایش دهیم
        cols_to_show = [
            "id",
            "title_fa",
            "is_view",
            "is_purchase",
            "conversion_rate",
        ]

        # فقط ستون‌هایی که واقعاً در جدول وجود دارند
        available_cols = [
            col
            for col in cols_to_show
            if col in anomaly_products.columns
        ]

        print(
            anomaly_products[available_cols]
            .head(10)
            .to_string(index=False)
        )

    else:
        print("✅ هیچ محصولی پیدا نشد که خرید آن بیشتر از بازدید باشد.")

else:
    print(
        "⚠️ ستون‌های 'is_view' یا 'is_purchase' در این فایل یافت نشدند."
    )

📊 تعداد کل محصولات بررسی شده: 948,352
❌ تعداد محصولاتی که خرید بیشتر از بازدید داشتند: 0

✅ هیچ محصولی پیدا نشد که خرید آن بیشتر از بازدید باشد.
